# 00 — ToS Scraper
**DS593 Final Project**

Scrapes Terms of Service and Privacy Policy documents from 20 major apps and saves as plain text files.
Each document is timestamped so the corpus can be refreshed at any time to stay current.


## Step 1: Install Dependencies

In [ ]:
!pip install requests beautifulsoup4 playwright -q
!playwright install chromium
!playwright install-deps chromium
print("✓ Dependencies installed")


## Step 2: Imports + Config

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import os
import json
from datetime import datetime
from playwright.sync_api import sync_playwright

OUTPUT_DIR  = "tos_corpus"
SCRAPE_DATE = datetime.today().strftime("%Y-%m-%d")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}/")
print(f"✓ Scrape date: {SCRAPE_DATE}")


## Step 3: Company List
Add or remove companies here. Each entry needs a name, ToS URL, and optional Privacy Policy URL.


In [ ]:
COMPANIES = [
    {
        "name": "spotify",
        "tos_url": "https://www.spotify.com/us/legal/end-user-agreement/",
        "privacy_url": "https://www.spotify.com/us/legal/privacy-policy/",
    },
    {
        "name": "instagram",
        "tos_url": "https://www.instagram.com/legal/terms/",
        "privacy_url": "https://privacycenter.instagram.com/policy",
    },
    {
        "name": "tiktok",
        "tos_url": "https://www.tiktok.com/legal/page/us/terms-of-service/en",
        "privacy_url": "https://www.tiktok.com/legal/page/us/privacy-policy/en",
    },
    {
        "name": "venmo",
        "tos_url": "https://venmo.com/legal/us-user-agreement/",
        "privacy_url": "https://venmo.com/legal/us-privacy-policy/",
    },
    {
        "name": "airbnb",
        "tos_url": "https://www.airbnb.com/help/article/2908",
        "privacy_url": "https://www.airbnb.com/help/article/2855",
    },
    {
        "name": "uber",
        "tos_url": "https://www.uber.com/legal/en/document/?name=general-terms-of-use&country=united-states&lang=en",
        "privacy_url": "https://www.uber.com/legal/en/document/?name=privacy-notice&country=united-states&lang=en",
    },
    {
        "name": "netflix",
        "tos_url": "https://help.netflix.com/legal/termsofuse",
        "privacy_url": "https://help.netflix.com/legal/privacy",
    },
    {
        "name": "twitter_x",
        "tos_url": "https://twitter.com/en/tos",
        "privacy_url": "https://twitter.com/en/privacy",
    },
    {
        "name": "google",
        "tos_url": "https://policies.google.com/terms",
        "privacy_url": "https://policies.google.com/privacy",
    },
    {
        "name": "facebook",
        "tos_url": "https://www.facebook.com/terms.php",
        "privacy_url": "https://www.facebook.com/privacy/policy/",
    },
    {
        "name": "amazon",
        "tos_url": "https://www.amazon.com/gp/help/customer/display.html?nodeId=GLSBYFE9MGKKQXXM",
        "privacy_url": "https://www.amazon.com/gp/help/customer/display.html?nodeId=GX7NJQ4ZB8MHFRNJ",
    },
    {
        "name": "paypal",
        "tos_url": "https://www.paypal.com/us/legalhub/useragreement-full",
        "privacy_url": "https://www.paypal.com/us/legalhub/privacy-full",
    },
    {
        "name": "discord",
        "tos_url": "https://discord.com/terms",
        "privacy_url": "https://discord.com/privacy",
    },
    {
        "name": "snapchat",
        "tos_url": "https://snap.com/en-US/terms",
        "privacy_url": "https://snap.com/en-US/privacy/privacy-policy",
    },
    {
        "name": "youtube",
        "tos_url": "https://www.youtube.com/t/terms",
        "privacy_url": "https://policies.google.com/privacy",
    },
    {
        "name": "doordash",
        "tos_url": "https://www.doordash.com/terms/",
        "privacy_url": "https://www.doordash.com/privacy/",
    },
    {
        "name": "robinhood",
        "tos_url": "https://robinhood.com/us/en/about/legal/",
        "privacy_url": "https://robinhood.com/us/en/about/privacy/",
    },
    {
        "name": "coinbase",
        "tos_url": "https://www.coinbase.com/legal/user_agreement/united_states",
        "privacy_url": "https://www.coinbase.com/legal/privacy",
    },
    {
        "name": "tinder",
        "tos_url": "https://policies.tinder.com/terms/intl/en",
        "privacy_url": "https://policies.tinder.com/privacy/intl/en",
    },
    {
        "name": "linkedin",
        "tos_url": "https://www.linkedin.com/legal/user-agreement",
        "privacy_url": "https://www.linkedin.com/legal/privacy-policy",
    },
]

print(f"✓ {len(COMPANIES)} companies loaded")


## Step 4: Scraper Functions
Two strategies:
- **requests + BeautifulSoup** — fast, works for most static pages
- **Playwright** — headless browser fallback for JS-rendered pages


In [ ]:
def clean_html(soup):
    """Remove boilerplate tags and return clean plain text."""
    for tag in soup(["script", "style", "nav", "footer", "header", "img", "button"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


def scrape_requests(url):
    """Try scraping with requests — fast and simple."""
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        return clean_html(soup)
    except Exception as e:
        print(f"    requests failed: {e}")
        return ""


def scrape_playwright(url):
    """Fallback: headless browser for JS-rendered pages."""
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page()
            page.goto(url, timeout=30000, wait_until="networkidle")
            html = page.content()
            browser.close()
        soup = BeautifulSoup(html, "html.parser")
        return clean_html(soup)
    except Exception as e:
        print(f"    playwright failed: {e}")
        return ""


def scrape_url(url):
    """Try requests first, fall back to Playwright if result is too short."""
    print(f"    Trying requests...")
    text = scrape_requests(url)
    if len(text) >= 1000:
        return text
    print(f"    Too short ({len(text)} chars), trying Playwright...")
    return scrape_playwright(url)


def save_document(company, doc_type, text):
    """Save scraped text to output directory."""
    filename = f"{company}_{doc_type}_{SCRAPE_DATE}.txt"
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"    ✓ Saved: {filename} ({len(text):,} chars)")
    return filepath


print("✓ Scraper functions ready")


## Step 5: Run Scraper
Scrapes all companies and saves to `tos_corpus/`.
Expect this to take 5-10 minutes for 20 companies.


In [ ]:
manifest = []
failed   = []

for company in COMPANIES:
    name = company["name"]
    print(f"\n── {name.upper()} ──")

    for doc_type, url in [("tos", company.get("tos_url")), ("privacy", company.get("privacy_url"))]:
        if not url:
            continue

        print(f"  [{doc_type.upper()}] {url}")
        text = scrape_url(url)

        if not text or len(text) < 500:
            print(f"    ✗ Failed or too short — add manually if needed")
            failed.append({"company": name, "doc_type": doc_type, "url": url})
            continue

        filepath = save_document(name, doc_type, text)
        manifest.append({
            "company":    name,
            "doc_type":   doc_type,
            "url":        url,
            "filepath":   filepath,
            "scrape_date": SCRAPE_DATE,
            "char_count": len(text),
        })

        time.sleep(2)  # polite delay

# Save manifest
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n{'='*50}")
print(f"✓ Done. {len(manifest)} documents saved.")
if failed:
    print(f"✗ {len(failed)} failed — copy-paste these manually:")
    for item in failed:
        print(f"  {item['company']} ({item['doc_type']}): {item['url']}")
print(f"✓ Manifest saved to {manifest_path}")


## Step 6: Verify Output

In [ ]:
import pandas as pd

df = pd.DataFrame(manifest)
print(f"Total documents scraped: {len(df)}")
print(f"Total characters:        {df['char_count'].sum():,}\n")
print(df[["company", "doc_type", "char_count"]].to_string(index=False))


## Step 7: Manual Fallback (for failed sites)

For any sites that failed scraping:
1. Open the URL in your browser
2. Select all text (Cmd+A) and copy
3. Paste into a new `.txt` file
4. Save to `tos_corpus/` with naming: `{company}_{doc_type}_{date}.txt`
   e.g. `facebook_tos_2026-04-17.txt`
5. Run the cell below to add it to the manifest


In [ ]:
# After adding manual files, run this to verify everything in the corpus folder
all_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")])
print(f"Files in tos_corpus/ ({len(all_files)} total):\n")
for fname in all_files:
    path = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(path)
    print(f"  {fname:50s} {size:>10,} bytes")
